
# Machine Learning aplicado, Clustering 1
## Segmentar la base de clientes de un banco: k-means desde cero

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos

Hasta ahora todos los problemas del curso tenían una **respuesta correcta** guardada en una columna: el monto que compró
el cliente, si pagó o no pagó el crédito. El modelo aprendía a predecir esa columna y lo medíamos comparando sus
predicciones con la verdad en testing. Eso es aprendizaje **supervisado**.

Hoy la columna de respuesta no existe. Un banco tiene 5.660 clientes con su edad, ocupación, estado civil, educación y
los productos que tienen contratados, y la pregunta es otra: **¿qué tipos de clientes hay?** Nadie nos dice cuántos tipos
son ni cómo se llaman. Eso es aprendizaje **no supervisado**, y la herramienta más usada para esta pregunta es el
**clustering**: encontrar grupos de clientes parecidos entre sí y distintos de los demás.

El cambio de pregunta cambia todo lo demás. No hay testing que diga si el modelo acertó, así que la evaluación es
distinta (y más incómoda); el algoritmo trabaja con **distancias** entre clientes, así que la forma en que convertimos
una ocupación o un estado civil en números decide el resultado; y al final el modelo entrega grupos que **alguien tiene
que interpretar y ponerle nombre**, porque el algoritmo no sabe qué es un jubilado.

La idea central del notebook es esa conversión. Un algoritmo de clustering no entiende que un cliente casado se parece
más a uno divorciado que a uno soltero, ni que un gerente y un técnico comparten más que un gerente y un estudiante.
Esa similitud, que para cualquier persona del banco es obvia, hay que **escribirla a mano**: traducir cada categoría a
un puntaje sobre un eje con sentido de negocio, declarando los supuestos y manteniéndolos consistentes de principio a
fin. Es trabajo humano, no un paso técnico, y es lo que después permite leer los segmentos con esa misma connotación.

### Qué vas a aprender hoy

1. Qué problema de negocio resuelve una segmentación y qué hace un banco con ella.
2. Cómo funciona k-means, paso a paso, y por qué lo único que ve son distancias.
3. El preprocesamiento que decide el resultado: binarias a 0/1, qué hacer con "Desconocido", categóricas a puntajes sobre ejes de negocio declarados (y por qué el one-hot a ciegas borra la similitud), y escalar.
4. Cuántos grupos pedir: la inercia (el codo) y la silueta, y qué miden de verdad.
5. Cómo se interpreta un segmento: centroides en los ejes que definimos y en unidades originales, composición, coordenadas paralelas, y las pruebas estadísticas que confirman que los grupos difieren.
6. Qué pasa cuando se codifica a ciegas con one-hot, y cuánto dependen los segmentos de los supuestos del mapeo.

Los pasos de este notebook se pueden recorrer también en el sitio interactivo del curso (pestaña "Segmentación de
clientes"), y el **Clustering Playground** del hub permite mover los centroides a mano para ver cómo k-means converge.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
SEMILLA = 7



## 1. El problema: 5.660 clientes y ninguna etiqueta

Un banco de personas tiene una base de clientes con lo que sabe de cada uno: cuántos años tiene, en qué trabaja, su
estado civil, su nivel de educación, y si tiene cuenta de ahorro, crédito hipotecario o crédito de consumo. Con esa
base el banco quiere hacer tres cosas, y las tres necesitan **grupos**:

- **Ofertas**: no tiene sentido ofrecerle un hipotecario a quien ya tiene uno ni un plan de ahorro para la universidad a
  un jubilado. Con segmentos se diseñan pocas campañas, una por grupo, en vez de una por cliente.
- **Canal y lenguaje**: un estudiante de 22 años y un gerente de 55 no se comunican por el mismo canal ni con el mismo
  tono.
- **Riesgo y provisiones**: los segmentos sirven para mirar la mora, la rentabilidad y el abandono por tipo de cliente,
  antes de construir un modelo predictivo.

Lo que no tiene el banco es una columna que diga "este cliente es del tipo 3". Esa columna la vamos a construir.

Las columnas vienen con nombres en español; las renombramos para que sean fáciles de escribir. Si tienes el archivo del
curso (`base_clientes_banco.xlsx`), pega su link en `URL_DATOS`; si queda vacío, se baja de la copia del curso.


In [ ]:

URL_DATOS = ""    # <-- pega aquí el link del archivo del curso (base_clientes_banco.xlsx) si lo tienes
URL_RESPALDO = "https://www.dropbox.com/s/lfjqcx9uoey6bap/base_clientes_banco.xlsx?dl=1"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

clientes = pd.read_excel(normalizar_link(URL_DATOS or URL_RESPALDO))
clientes.columns = ["edad", "ocupacion", "estado_civil", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo", "impago"]
print("Clientes:", len(clientes), "  Columnas:", clientes.shape[1])
clientes.head(8)



Antes de modelar hay que mirar qué hay en cada columna. Solo `edad` es un número; las otras siete son texto, y en
varias aparece la categoría **"Desconocido"**, que no es una categoría más sino un dato faltante con disfraz.


In [ ]:

print(clientes["edad"].describe().round(1).to_string())
print()
for col in clientes.columns[1:]:
    print(f"{col}: " + ", ".join(f"{k} ({v})" for k, v in clientes[col].value_counts().items()))



Tres cosas que esta tabla ya decide:

- **`impago`** tiene un solo "Si" entre 5.660 clientes y 1.035 desconocidos. Una columna donde todos valen lo mismo no
  separa a nadie, y además es exactamente el tipo de columna que uno querría **predecir** con los segmentos, no usar
  para armarlos. La sacamos.
- **`hipotecario`** y **`credito_consumo`** tienen 115 desconocidos cada una (2%). Son pocos; los vamos a tratar como
  faltantes.
- **`educacion`** tiene un orden natural (sin educación, básica, media, curso profesional, universitaria). **`ocupacion`** y
  **`estado_civil`** no tienen un orden único, pero sí tienen **similitudes** que cualquier persona del banco reconoce: un
  obrero y un empleado de servicios se parecen más entre sí que un obrero y un gerente. Hacer explícitas esas similitudes
  es el centro de la sección 3.



## 2. Qué hace k-means

K-means (MacQueen, 1967; el algoritmo iterativo es de Lloyd, 1957) busca $K$ puntos, los **centroides**, de modo que
cada cliente quede asignado al centroide más cercano y la suma de las distancias al cuadrado de cada cliente a su
centroide, la **inercia** (en la literatura, *within-cluster sum of squares*, suma de cuadrados dentro de los grupos),
sea lo más chica posible:

$$\text{inercia} = \sum_{k=1}^{K} \sum_{i \in C_k} \lVert x_i - \mu_k \rVert^2$$

Cada símbolo: $K$ es el número de grupos, que nosotros fijamos; $C_k$ es el conjunto de clientes asignados al grupo
$k$; $x_i$ es el vector de características del cliente $i$ (una fila de la matriz numérica que vamos a construir en la
sección 3, ya escalada); $\mu_k$ es el centroide del grupo $k$, el promedio de los vectores de sus clientes; y
$\lVert \cdot \rVert^2$ es la distancia euclidiana al cuadrado, la suma de las diferencias al cuadrado columna por
columna. En palabras: la inercia es cuánto se alejan los clientes, en total, del centro de su propio grupo. En el código
la inercia es `modelo.inertia_`, los centroides `modelo.cluster_centers_` y la asignación `modelo.labels_`. El algoritmo
es de una simpleza que sorprende:

1. Elegir $K$ centroides iniciales (al azar, o con el truco `k-means++` que los reparte lejos entre sí).
2. **Asignar** cada cliente al centroide más cercano.
3. **Mover** cada centroide al promedio de los clientes que le tocaron.
4. Repetir 2 y 3 hasta que nadie cambie de grupo.

Cada vuelta baja la inercia o la deja igual, así que converge; pero converge a un mínimo **local** que depende del
punto de partida, y por eso scikit-learn lo corre varias veces (`n_init`) y se queda con la mejor.

Lo que hay que retener es la frase del paso 2: **el centroide más cercano**. K-means no sabe qué es una ocupación ni una
edad; solo calcula distancias euclidianas entre filas de números. Entonces todo lo que decidamos sobre cómo convertir
"Obrero" o "Casado" en números, y sobre la escala de cada columna, se traduce directamente en qué clientes quedan juntos.
Por eso la sección que sigue es la más importante del notebook.



## 3. Preprocesamiento: convertir clientes en vectores sin mentir

### 3.1 Las binarias: sí / no a 1 / 0, y "Desconocido" como faltante

Una columna con dos valores se convierte en 0 y 1 sin perder nada. El "Desconocido" de `hipotecario` y
`credito_consumo` lo marcamos como faltante y lo imputamos con el valor más frecuente (la moda), que es lo razonable
cuando son el 2% de las filas; con muchos más convendría agregar una columna indicadora "se desconoce".


In [ ]:

datos = clientes.drop(columns="impago").copy()
for col in ["cuenta_ahorro", "hipotecario", "credito_consumo"]:
    datos[col] = datos[col].map({"Si": 1, "No": 0})          # "Desconocido" queda como NaN
print("Faltantes por columna después de convertir las binarias:")
print(datos.isna().sum().to_string())
for col in ["hipotecario", "credito_consumo"]:
    datos[col] = datos[col].fillna(datos[col].mode()[0]).astype(int)
datos["cuenta_ahorro"] = datos["cuenta_ahorro"].astype(int)



### 3.2 Las categóricas: traducir similitud a números, con lógica de negocio declarada

Esta es la decisión más importante del notebook, y la que más veces se hace mal.

La receta automática para una variable de texto es el **one-hot encoding** (variables dummy): una columna 0/1 por
categoría. "Casado" pasa a ser (1, 0, 0), "Divorciado" (0, 1, 0) y "Soltero" (0, 0, 1). Es rápido, no exige pensar, y
tiene un costo que no se ve hasta que se miran los segmentos: **todas las categorías quedan a la misma distancia entre
sí**. La distancia euclidiana entre dos vectores one-hot distintos es siempre $\sqrt{2}$, cualquiera sea el par. Para
k-means, un casado está exactamente tan lejos de un divorciado como de un soltero; un gerente tan lejos de un técnico
como de un estudiante; y si la variable fuera la comuna, Providencia estaría tan lejos de Las Condes como de Puerto
Montt. Toda la información de **similitud** entre valores de una misma columna, que para una persona del banco es
obvia, se pierde. Y además la variable pasa a ocupar tantas dimensiones como categorías tenga, lo que trae otros
problemas que veremos en la sección 6.

La alternativa es hacer el trabajo que el algoritmo no puede hacer: definir **un eje con sentido de negocio** y asignar
a cada categoría un **puntaje** sobre ese eje. Para el estado civil, un eje razonable es el grado de compromiso
familiar del hogar, de 0 a 100: Soltero 0, Divorciado 50, Casado 100. Con eso le decimos a k-means lo que sabemos:
que un divorciado está a mitad de camino (tuvo un hogar, probablemente tiene hijos y obligaciones, y hoy decide solo),
en vez de dejarlo equidistante de todo. El valor 50 no sale de los datos; sale de nuestra lectura del negocio, y por
eso hay que **escribirlo y defenderlo**.

Para la ocupación un solo eje no alcanza, y conviene darse cuenta antes de codificar: un jubilado y un estudiante
tienen ambos ingresos bajos, pero el del jubilado es fijo y seguro y el del estudiante es casi inexistente e inestable.
Si los pusiéramos juntos en un único eje de "ingreso", el algoritmo los vería iguales. Usamos entonces **dos ejes**,
cada uno de 0 a 100:

- **Nivel de ingreso esperado** de la ocupación: Gerente 100, Emprendedor 75, Empleado Independiente 60, Técnico 60,
  Administrativo 50, Servicios 35, Obrero 30, Jubilado 30, Dueña de Casa 10, Estudiante 10, Desempleado 0.
- **Estabilidad del ingreso** (qué tan previsible y formal es): Jubilado 100 (una pensión no falla), Administrativo 85,
  Gerente 85, Técnico 80, Obrero 60, Servicios 55, Empleado Independiente 40, Emprendedor 35, Dueña de Casa 30,
  Estudiante 15, Desempleado 0.

Doce categorías pasan a ser **dos** números por cliente, con una lectura clara, en vez de doce columnas sin relación
entre sí. La educación, que ya tiene orden, va al mismo tipo de escala para que todo quede en las mismas unidades:
Sin Educación 0, Básica 25, Media 50, Curso Profesional 75, Grado Universitario 100 (con esto suponemos que cada
salto educacional vale lo mismo, otro supuesto que queda declarado).

Tres reglas para que este mapeo sea un análisis y no una opinión suelta:

1. **Declarar los supuestos** en una tabla, antes de correr nada, con la lógica de cada eje. Es lo que hacemos en la
   celda siguiente.
2. **Mantenerlos consistentes** durante todo el análisis: los mismos puntajes para segmentar, para interpretar los
   centroides y para asignar clientes nuevos. Cambiar la escala a mitad de camino invalida todo lo anterior.
3. **Leer los resultados con esa connotación**: si un segmento tiene "estabilidad 80", eso significa "alto en el eje de
   estabilidad que nosotros definimos", no un hecho medido. Y en la sección 6 vamos a medir cuánto cambian los
   segmentos cuando cambia un supuesto, que es la forma honesta de saber de qué depende la segmentación.

Los "Desconocido" de las tres variables (37 en ocupación, 13 en estado civil, 282 en educación) son faltantes y se
imputan con la mediana del puntaje, que los deja en el centro del eje, a la misma distancia de los extremos.


In [ ]:

# Tabla de supuestos: cada categoría recibe un puntaje de 0 a 100 sobre un eje de negocio declarado.
# Estos puntajes son decisiones nuestras, no datos; se usan sin cambios durante todo el notebook.
PUNTAJE_INGRESO = {"Gerente": 100, "Emprendedor": 75, "Empleado Independiente": 60, "Técnico": 60, "Administrativo": 50,
                   "Servicios": 35, "Obrero": 30, "Jubilado": 30, "Dueña de Casa": 10, "Estudiante": 10, "Desempleado": 0}
PUNTAJE_ESTABILIDAD = {"Jubilado": 100, "Administrativo": 85, "Gerente": 85, "Técnico": 80, "Obrero": 60, "Servicios": 55,
                       "Empleado Independiente": 40, "Emprendedor": 35, "Dueña de Casa": 30, "Estudiante": 15, "Desempleado": 0}
PUNTAJE_COMPROMISO = {"Soltero": 0, "Divorciado": 50, "Casado": 100}
PUNTAJE_EDUCACION = {"Sin Educación": 0, "Básica": 25, "Media": 50, "Curso Profesional": 75, "Grado Universitario": 100}

supuestos = pd.DataFrame({"eje": ["ingreso", "estabilidad", "compromiso", "educacion"],
                          "variable_original": ["ocupacion", "ocupacion", "estado_civil", "educacion"],
                          "que_mide": ["nivel de ingreso esperado de la ocupación", "qué tan previsible y formal es ese ingreso",
                                       "grado de compromiso familiar del hogar", "años de educación formal, saltos iguales"],
                          "extremos": ["Desempleado 0 ... Gerente 100", "Desempleado 0 ... Jubilado 100", "Soltero 0, Divorciado 50, Casado 100",
                                       "Sin Educación 0 ... Universitario 100"]})
print(supuestos.to_string(index=False))

datos["ingreso"] = datos["ocupacion"].map(PUNTAJE_INGRESO)              # "Desconocido" queda NaN
datos["estabilidad"] = datos["ocupacion"].map(PUNTAJE_ESTABILIDAD)
datos["compromiso"] = datos["estado_civil"].map(PUNTAJE_COMPROMISO)
datos["educacion"] = datos["educacion"].map(PUNTAJE_EDUCACION)
for col in ["ingreso", "estabilidad", "compromiso", "educacion"]:
    datos[col] = datos[col].fillna(datos[col].median())

COLUMNAS_MODELO = ["edad", "ingreso", "estabilidad", "compromiso", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]
X = datos[COLUMNAS_MODELO].astype(float)
print()
print("Matriz para k-means:", X.shape[0], "clientes x", X.shape[1], "columnas")
X.head()



Antes de seguir, conviene ver con números qué cambió. La celda siguiente calcula la distancia entre pares de clientes
que solo difieren en una categoría, con las dos codificaciones. Con one-hot todas las distancias entre categorías
distintas valen lo mismo; con el mapeo, la distancia refleja la similitud que declaramos.


In [ ]:

def distancia_onehot(a, b):
    return 0.0 if a == b else np.sqrt(2)

pares = [("Casado", "Divorciado"), ("Casado", "Soltero"), ("Divorciado", "Soltero")]
print("Estado civil: distancia entre dos clientes idénticos salvo en esta columna")
for a, b in pares:
    print(f"  {a:10s} vs {b:10s}   one-hot: {distancia_onehot(a, b):.2f}   mapeo (eje compromiso): {abs(PUNTAJE_COMPROMISO[a] - PUNTAJE_COMPROMISO[b]):>5.0f}")
print()
pares = [("Gerente", "Técnico"), ("Gerente", "Estudiante"), ("Obrero", "Servicios"), ("Jubilado", "Estudiante"), ("Jubilado", "Obrero")]
print("Ocupación: lo mismo, con los dos ejes (distancia euclidiana en el plano ingreso x estabilidad)")
for a, b in pares:
    d = np.hypot(PUNTAJE_INGRESO[a] - PUNTAJE_INGRESO[b], PUNTAJE_ESTABILIDAD[a] - PUNTAJE_ESTABILIDAD[b])
    print(f"  {a:10s} vs {b:10s}   one-hot: {distancia_onehot(a, b):.2f}   mapeo: {d:>6.1f}")

ocupaciones = pd.DataFrame({"ocupacion": list(PUNTAJE_INGRESO), "ingreso": list(PUNTAJE_INGRESO.values()),
                            "estabilidad": [PUNTAJE_ESTABILIDAD[o] for o in PUNTAJE_INGRESO],
                            "clientes": [int((clientes["ocupacion"] == o).sum()) for o in PUNTAJE_INGRESO]})
fig = px.scatter(ocupaciones, x="ingreso", y="estabilidad", size="clientes", text="ocupacion", size_max=45,
                 title="El mapa de ocupaciones que le entregamos a k-means (tamaño = número de clientes)")
fig.update_traces(textposition="top center")
fig.update_layout(height=480, width=760, xaxis_range=[-10, 115], yaxis_range=[-10, 115])
fig.show()



El gráfico es la segmentación de ocupaciones que **nosotros** hicimos antes de que corriera ningún algoritmo, y
conviene mirarlo con esa conciencia: k-means va a heredar estas cercanías. Jubilado y estudiante, que one-hot ponía a
la misma distancia que cualquier otro par, quedan ahora en esquinas opuestas del eje de estabilidad; obrero y servicios
quedan casi pegados. Si alguien del banco discrepa de un puntaje, este es el momento de discutirlo, no después de ver
los segmentos.



### 3.3 Escalar: que ninguna columna mande por su unidad

La edad va de 17 a 92, los puntajes de 0 a 100 y los productos de 0 a 1. Sin hacer nada, una diferencia de 50 puntos
de ingreso pesaría en la distancia 2.500 veces más (al cuadrado) que tener o no una cuenta de ahorro, por puro accidente
de unidades. Estandarizamos cada columna (restar su media y dividir por su desviación) para que todas pesen parecido:
$z_{ij} = (x_{ij} - \bar{x}_j) / s_j$, donde $x_{ij}$ es el valor del cliente $i$ en la columna $j$, $\bar{x}_j$ el
promedio de la columna y $s_j$ su desviación estándar muestral. Un $z$ de +2 se lee "dos desviaciones por encima del
cliente promedio". Fíjate en que escalar no cambia las cercanías **dentro** de un eje: divorciado sigue a mitad de camino
entre soltero y casado; solo cambia cuánto pesa ese eje frente a los demás. Es el mismo `StandardScaler` de los
notebooks de regresión, con una diferencia: aquí no hay training y testing que separar, porque no vamos a predecir nada
sobre clientes nuevos (si más adelante hubiera que asignar clientes nuevos a los segmentos, se usaría el escalador ya
ajustado, igual que antes).


In [ ]:

escalador = StandardScaler().fit(X)
Z = pd.DataFrame(escalador.transform(X), columns=X.columns, index=X.index)
print("Después de escalar, cada columna tiene media 0 y desviación 1:")
print(Z.describe().loc[["mean", "std"]].round(2).iloc[:, :8].to_string())



## 4. Cuántos grupos: el codo y la silueta

K-means necesita que le digamos $K$, y no hay una respuesta correcta escondida en los datos. Hay dos criterios que
ayudan, y conviene saber qué mide cada uno.

**La inercia** siempre baja cuando sube $K$ (con $K$ igual al número de clientes vale cero), así que no se puede
elegir el mínimo. Lo que se busca es el **codo**: el punto a partir del cual agregar un grupo más ya casi no la reduce.

**La silueta** (Rousseeuw, 1987) de un cliente compara qué tan cerca está de su propio grupo contra qué tan cerca está
del grupo vecino más próximo:

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$

donde $s_i$ es la silueta del cliente $i$, $a_i$ es la distancia promedio del cliente $i$ a los demás clientes de su
propio grupo (qué tan compacto es su grupo visto desde él) y $b_i$ la distancia promedio a los clientes del grupo más
cercano que no es el suyo (qué tan lejos queda la alternativa). El máximo del denominador deja el cociente entre -1 y 1.
En palabras: cuánto más cerca está el cliente de su grupo que del grupo vecino, en proporción. Vale cerca de 1 si está claramente en el grupo correcto, cerca de 0 si está en la frontera, y
negativo si probablemente está en el grupo equivocado. El promedio sobre todos los clientes es la silueta del
agrupamiento, y aquí sí se busca el **máximo**. Como referencia, sobre 0,5 se habla de estructura razonable y bajo 0,25
de estructura débil; con datos de clientes reales rara vez se pasa de 0,3, y no es un fracaso: significa que los
clientes forman un continuo, no islas.


In [ ]:

ks = range(2, 11)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Z, km.labels_, sample_size=3000, random_state=SEMILLA))
criterios = pd.DataFrame({"K": list(ks), "inercia": inercias, "silueta": siluetas}).set_index("K")
print(criterios.round(3).to_string())

fig = go.Figure()
fig.add_scatter(x=criterios.index, y=criterios["inercia"], mode="lines+markers", name="inercia")
fig.add_scatter(x=criterios.index, y=criterios["silueta"], mode="lines+markers", name="silueta", yaxis="y2")
fig.update_layout(title="Cuántos grupos: inercia (buscar el codo) y silueta (buscar el máximo)", xaxis_title="K",
                  yaxis=dict(title="inercia"), yaxis2=dict(title="silueta", overlaying="y", side="right"), height=420, width=760)
fig.show()



La inercia baja de forma pareja y no muestra un codo nítido; la silueta sube rápido hasta $K = 5$ (de 0,17 a 0,20) y
después oscila entre 0,19 y 0,21 hasta $K = 10$, sin un máximo claro. Es la situación típica: los criterios orientan, no deciden. En la
práctica el número de segmentos también se elige por lo que el negocio puede manejar: cinco campañas distintas son
operables, quince no. Vamos a trabajar con **K = 5**.


In [ ]:

K = 5
modelo = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z)
datos["segmento"] = modelo.labels_
clientes["segmento"] = modelo.labels_
print("Inercia:", round(modelo.inertia_), "  Silueta:", round(silhouette_score(Z, modelo.labels_, sample_size=3000, random_state=SEMILLA), 3))
print()
print("Clientes por segmento:")
print(datos["segmento"].value_counts().sort_index().to_string())



## 5. Interpretar los segmentos

El modelo entregó una etiqueta por cliente y nada más. Interpretar es volver a las unidades que entendemos: para cada
segmento, la edad promedio, el puntaje promedio en cada eje que definimos (ingreso, estabilidad, compromiso,
educación), el porcentaje que tiene cada producto, y la composición por ocupación y estado civil originales. Los
centroides en la escala estandarizada sirven para ver qué columna "tira" de cada grupo (un valor de +2 significa dos
desviaciones sobre el promedio general), pero el nombre del segmento sale de las unidades reales.

Y aquí se cobra el trabajo de la sección 3: como los ejes los definimos nosotros, los segmentos se leen **en esos
términos**. "Ingreso 63, estabilidad 79" no es un dato medido; significa "ocupaciones que nosotros pusimos arriba en
ingreso y en estabilidad". La interpretación hereda los supuestos, y eso es lo que la hace defendible frente a la
gerencia: cada número de la tabla se puede rastrear hasta una decisión declarada.


In [ ]:

perfil = datos.groupby("segmento").agg(clientes=("edad", "size"), edad=("edad", "mean"), ingreso=("ingreso", "mean"), estabilidad=("estabilidad", "mean"),
                                       compromiso=("compromiso", "mean"), educacion=("educacion", "mean"),
                                       pct_ahorro=("cuenta_ahorro", "mean"), pct_hipotecario=("hipotecario", "mean"), pct_consumo=("credito_consumo", "mean"))
perfil["pct_del_total"] = perfil["clientes"] / len(datos)
print("Perfil de cada segmento (puntajes de 0 a 100 en los ejes definidos; productos como fracción):")
print(perfil.round(2).to_string())
print()
for col in ["ocupacion", "estado_civil", "educacion"]:
    print(f"Composición por {col} (fila = segmento, fracción de sus clientes):")
    print(pd.crosstab(clientes["segmento"], clientes[col], normalize="index").round(2).to_string())
    print()



Lectura de los cinco segmentos (los números de segmento son arbitrarios y pueden cambiar de orden al correr de nuevo;
lo que importa es el perfil):

- **Sin ingreso propio** (402 clientes, 7%): 33 años, ingreso 6 y estabilidad 11 en nuestros ejes, es decir,
  estudiantes (46%), desempleados (41%) y dueñas de casa (12%), mayoritariamente solteros, y con la **mayor tenencia de
  cuenta de ahorro** (56%). Un grupo que one-hot nunca habría formado, porque junta tres ocupaciones distintas que
  comparten una situación económica.
- **Profesionales casados** (1.415, 25%): 42 años, ingreso 63, estabilidad 79, compromiso 92, la educación más alta
  (87): administrativos, técnicos y gerentes con hogar formado. El segmento de hipotecario y productos de inversión.
- **Casados de ingreso bajo** (1.797, 32%): 45 años, ingreso 37 y la educación más baja (36): obreros (44%), servicios,
  administrativos de educación básica y jubilados. El más numeroso, y el de menor tenencia de cuenta de ahorro (22%).
- **Con crédito de consumo** (810, 14%): es el único segmento con crédito de consumo (100% contra 0% en los demás), y por
  lo demás una mezcla de todo. Lo definió una sola columna binaria, y en la sección 6 explicamos por qué.
- **Solteros con trabajo** (1.236, 22%): 33 años, compromiso 2 (96% solteros), ingreso y estabilidad medios, educación
  70: la base joven en formación, con 38% de cuenta de ahorro.

Compara esto con lo que habría salido sin el mapeo (lo vamos a correr en la sección 6): grupos que son una sola
ocupación cada uno. Acá los segmentos cruzan ocupaciones porque les dijimos qué ocupaciones se parecen.


In [ ]:

# Centroides en escala estandarizada: qué columnas definen a cada segmento
centroides = pd.DataFrame(modelo.cluster_centers_, columns=Z.columns)
centroides.insert(0, "segmento", range(K))
fig = px.parallel_coordinates(centroides, color="segmento", dimensions=list(Z.columns),
                              title="Centroides en unidades estandarizadas (0 = promedio general; +2 = dos desviaciones por encima)")
fig.update_layout(height=480)
fig.show()


In [ ]:

fig = px.scatter(datos.assign(segmento=datos["segmento"].astype(str), ocupacion=clientes["ocupacion"], estado_civil=clientes["estado_civil"],
                              ingreso_j=datos["ingreso"] + np.random.default_rng(0).uniform(-3, 3, len(datos)),
                              estabilidad_j=datos["estabilidad"] + np.random.default_rng(1).uniform(-3, 3, len(datos))),
                 x="ingreso_j", y="estabilidad_j", color="segmento", opacity=0.45, hover_data=["ocupacion", "estado_civil", "edad"],
                 labels={"ingreso_j": "ingreso (eje definido, con ruido para ver los puntos)", "estabilidad_j": "estabilidad (eje definido)"},
                 title="Los clientes en el plano de ocupación que definimos, coloreados por segmento")
fig.update_layout(height=500, width=820)
fig.show()
comp = pd.crosstab(clientes["segmento"], clientes["ocupacion"], normalize="index").reset_index().melt(id_vars="segmento", var_name="ocupacion", value_name="fraccion")
fig = px.bar(comp, x="segmento", y="fraccion", color="ocupacion", title="Composición de cada segmento por ocupación", barmode="stack")
fig.update_layout(height=420, width=820)
fig.show()



### Las pruebas: ¿los segmentos difieren de verdad?

El gráfico sugiere diferencias; la estadística las confirma. Para una variable numérica como la edad o un puntaje, el
**ANOVA de un factor** (Fisher, 1925) compara las medias entre los $K$ segmentos con el estadístico $F$, el cociente
entre la varianza de las medias de los grupos y la varianza dentro de los grupos: la hipótesis nula es que todas las
medias son iguales, y un valor p pequeño la rechaza. Para una variable categórica como la ocupación original, la prueba
es **chi-cuadrado de independencia** (Pearson, 1900) sobre la tabla de contingencia segmento por categoría: compara las
frecuencias observadas con las que habría si la distribución de ocupaciones fuera la misma en todos los segmentos.

Un detalle de implementación que suele salir mal: `f_oneway` recibe **una muestra por grupo** (la edad de los clientes
del segmento 0, la del segmento 1, y así), no la columna completa junto con la etiqueta.

Y una advertencia honesta: con 5.660 clientes casi cualquier diferencia sale significativa, y además los grupos se
armaron precisamente para que difieran en estas columnas, así que un valor p chico aquí es casi inevitable. La prueba
sirve para detectar la columna que **no** separa (la que puede sacarse del análisis), más que para celebrar las que sí.


In [ ]:

filas = []
for col in COLUMNAS_MODELO:
    muestras = [datos.loc[datos["segmento"] == k, col] for k in range(K)]
    F, p = stats.f_oneway(*muestras)
    filas.append(dict(variable=col, prueba="ANOVA", estadistico=F, valor_p=p))
for col in ["ocupacion", "estado_civil", "educacion"]:
    tabla = pd.crosstab(clientes["segmento"], clientes[col])
    chi2, p, gl, _ = stats.chi2_contingency(tabla)
    filas.append(dict(variable=col + " (original)", prueba="chi-cuadrado", estadistico=chi2, valor_p=p))
pruebas = pd.DataFrame(filas)
pruebas["valor_p"] = pruebas["valor_p"].map(lambda v: f"{v:.2e}")
print(pruebas.round(1).to_string(index=False))



La única columna que no separa es `hipotecario` (valor p de 0,42): la tenencia de hipotecario es prácticamente la
misma en los cinco segmentos (56% a 60%). Para una campaña de hipotecarios estos segmentos no sirven tal como están,
y eso es información útil: habría que segmentar con otras columnas o directamente construir un modelo de propensión.



## 6. Qué pasa si se codifica a ciegas, y de qué dependen los segmentos

### 6.1 One-hot: los segmentos se vuelven categorías

Repetimos exactamente el mismo procedimiento (binarias, educación en escala, estandarizar, k-means con K = 5) pero
con `ocupacion` y `estado_civil` en one-hot, que es lo que hace cualquiera que no se detenga a pensar en la similitud.
La matriz pasa de 8 columnas a 19.


In [ ]:

X_onehot = pd.get_dummies(datos[["edad", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]].join(clientes[["ocupacion", "estado_civil"]]),
                          columns=["ocupacion", "estado_civil"], dtype=float)
X_onehot = X_onehot.drop(columns=[c for c in X_onehot.columns if c.endswith("_Desconocido")])
Z_onehot = StandardScaler().fit_transform(X_onehot)
modelo_onehot = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z_onehot)
clientes["segmento_onehot"] = modelo_onehot.labels_
from sklearn.metrics import adjusted_rand_score
print("Columnas:", X_onehot.shape[1], "  Silueta:", round(silhouette_score(Z_onehot, modelo_onehot.labels_, sample_size=3000, random_state=SEMILLA), 3),
      "  Rand ajustado contra la segmentación con mapeo:", round(adjusted_rand_score(clientes["segmento"], clientes["segmento_onehot"]), 3))
print()
print("Composición por ocupación de los segmentos one-hot:")
print(pd.crosstab(clientes["segmento_onehot"], clientes["ocupacion"], normalize="index").round(2).to_string())
print()
print("Composición por estado civil:")
print(pd.crosstab(clientes["segmento_onehot"], clientes["estado_civil"], normalize="index").round(2).to_string())



Tres de los cinco segmentos son **una sola categoría**: "los estudiantes" (100%), "los jubilados" (100%), "los obreros"
(91%). Los otros dos son el resto de la base partido por estado civil. La silueta es parecida (0,18 contra 0,20), así
que el criterio numérico no avisa; el problema se ve solo al mirar la composición. El índice de Rand ajustado
(Hubert y Arabie, 1985), que vale 1 si dos particiones coinciden y 0 si coinciden tanto como dos particiones al azar,
da 0,31: son segmentaciones distintas de la misma base.

Lo que pasó tiene dos partes. La primera es la que ya discutimos: one-hot borró la similitud, así que el algoritmo no
tenía cómo juntar a un estudiante con un desempleado ni a un obrero con un empleado de servicios. La segunda es
mecánica y vale la pena entenderla: una categoría rara, digamos "Estudiante" con 186 clientes de 5.660, se convierte en
una columna con 3% de unos. Al estandarizarla, ese 1 se transforma en un valor cercano a +5,4 (porque la desviación de
una columna tan desbalanceada es muy chica), mientras que el 0 queda cerca de -0,2. Los 186 clientes con ese 1 quedan
**lejísimos** de todos los demás en esa dimensión, y lo más barato en inercia es darles un centroide propio. Mientras
más categorías tenga una nominal, más dimensiones aporta, más manda, y más se parece el resultado a una tabla de
frecuencias que a una segmentación. Es la **maldición de la dimensionalidad** aplicada al clustering: con muchas
columnas 0/1 todos los puntos quedan lejos de todos y la distancia deja de discriminar.

El mismo mecanismo explica el segmento "con crédito de consumo" de nuestra segmentación: `credito_consumo` es una
binaria con 14% de unos, y estandarizada separa mucho a quienes lo tienen. Ahí la decisión es de negocio: si el banco
quiere que un producto defina un segmento (es accionable), se deja; si no, se saca de las columnas de segmentar y se
usa para describir, como hacemos en el ejercicio 2.

### 6.2 Sensibilidad: cuánto dependen los segmentos de nuestros supuestos

Si los puntajes son decisiones, la pregunta obligada es cuánto cambiaría el resultado con otras decisiones razonables.
Probamos tres cambios, uno a la vez, y medimos el Rand ajustado contra la segmentación base:

1. Poner al divorciado en 0 en vez de 50 (leer "divorciado" como "hoy decide solo", sin compromiso).
2. Subir el ingreso del jubilado de 30 a 60 (leer la jubilación como una etapa con patrimonio acumulado).
3. Sacar el eje de estabilidad y dejar la ocupación con un solo eje de ingreso.


In [ ]:

def segmentar(tabla, columnas, k=K):
    Zs = StandardScaler().fit_transform(tabla[columnas].astype(float))
    return KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit(Zs).labels_

variantes = {}
d1 = datos.copy(); d1["compromiso"] = clientes["estado_civil"].map({"Soltero": 0, "Divorciado": 0, "Casado": 100}).fillna(50)
variantes["divorciado = 0"] = segmentar(d1, COLUMNAS_MODELO)
d2 = datos.copy(); d2["ingreso"] = clientes["ocupacion"].map({**PUNTAJE_INGRESO, "Jubilado": 60}).fillna(datos["ingreso"].median())
variantes["jubilado ingreso 60"] = segmentar(d2, COLUMNAS_MODELO)
variantes["sin eje estabilidad"] = segmentar(datos, [c for c in COLUMNAS_MODELO if c != "estabilidad"])

sensibilidad = pd.DataFrame({"variante": list(variantes), "rand_ajustado_vs_base": [adjusted_rand_score(datos["segmento"], v) for v in variantes.values()]})
print(sensibilidad.round(3).to_string(index=False))
print()
print("Qué pasa con los jubilados cuando su ingreso sube a 60 (fila = segmento base, columna = segmento nuevo):")
jub = clientes["ocupacion"] == "Jubilado"
print(pd.crosstab(datos.loc[jub, "segmento"], variantes["jubilado ingreso 60"][jub.values]))



Los tres cambios mueven la segmentación (Rand ajustado entre 0,69 y 0,77: la mayoría de los clientes sigue con los
mismos compañeros, una minoría cambia de grupo), y cada uno la mueve donde se esperaría: mover al divorciado cambia
de grupo a los divorciados, subir el ingreso del jubilado reagrupa a 239 de los 295 jubilados en un mismo segmento nuevo
(la tabla de abajo lo muestra), y sacar el eje de estabilidad vuelve a mezclar estudiantes con jubilados. Eso es exactamente lo que queremos de una
segmentación con supuestos declarados: que se pueda decir "si el banco prefiere leer la jubilación de esta otra forma,
cambian estos 295 clientes y nada más". Con one-hot no hay supuestos que discutir, pero tampoco hay similitud: la
decisión no desaparece, se esconde.

## 7. Lo que hay que llevarse

1. Clustering es aprendizaje sin etiqueta: el modelo propone grupos y la interpretación la hacemos nosotros, en unidades que entendemos.
2. K-means solo ve distancias euclidianas. El preprocesamiento no es un trámite previo: es donde se decide el resultado.
3. Codificar una categórica con one-hot a ciegas borra toda la similitud entre sus valores (casado y divorciado quedan tan lejos como casado y soltero) y multiplica las dimensiones. En su lugar, definir ejes con sentido de negocio y traducir cada categoría a un puntaje: es trabajo humano, con supuestos que se declaran antes, se mantienen consistentes durante todo el análisis y se usan para leer los resultados.
4. Una variable puede necesitar más de un eje (la ocupación: ingreso y estabilidad). Pensar los ejes antes de codificar.
5. Binarias a 0/1; "Desconocido" es faltante, no categoría; escalar al final.
6. La inercia busca el codo y la silueta el máximo; los dos orientan y el negocio decide. Siluetas de 0,2 a 0,3 son normales con clientes reales, y una silueta parecida no significa una segmentación parecida.
7. Medir la sensibilidad a los supuestos con el índice de Rand ajustado: una buena segmentación cambia donde se cambió el supuesto, y en ningún otro lado.
8. ANOVA y chi-cuadrado confirman diferencias, pero con miles de clientes casi todo es significativo: úsalos para encontrar lo que no separa.



## Ejercicios

1. **Tu propio mapeo.** Propón otra tabla de puntajes para `ocupacion` (por ejemplo, un eje de "etapa de vida" en vez
   de estabilidad, o un orden distinto de ingresos), justifica cada valor en una frase, vuelve a segmentar con K = 5 y
   compara con la segmentación base usando el índice de Rand ajustado y la composición por ocupación. ¿Qué clientes
   cambiaron de grupo? ¿Es lo que esperabas dado tu cambio?
2. **Segmentar con unas columnas y describir con otras.** Saca `credito_consumo` y `hipotecario` de las columnas de
   segmentar (déjalos para describir), vuelve a correr k-means con K = 5 y mira si desaparece el segmento definido por
   el crédito de consumo. Luego calcula el porcentaje de crédito de consumo por segmento nuevo: ¿hay segmentos donde
   el producto esté naturalmente concentrado?
3. **Estabilidad.** Una segmentación sirve si no cambia cuando cambian un poco los datos. Parte la base al azar en dos
   mitades, corre k-means con K = 5 en cada una, asigna los clientes de la segunda mitad a los centroides de la primera
   y mide el índice de Rand ajustado entre esa asignación y la segmentación propia de la segunda mitad.



## Soluciones


In [ ]:

# Ejercicio 1: otro mapeo para la ocupación (eje "etapa de vida" en lugar de estabilidad)
# Supuesto: la etapa de vida ordena las ocupaciones por el momento típico de la carrera, de 0 (formación) a 100 (retiro).
PUNTAJE_ETAPA = {"Estudiante": 0, "Desempleado": 30, "Servicios": 35, "Obrero": 45, "Administrativo": 45, "Técnico": 50, "Empleado Independiente": 55,
                 "Dueña de Casa": 55, "Emprendedor": 60, "Gerente": 70, "Jubilado": 100}
d_ej = datos.copy()
d_ej["etapa"] = clientes["ocupacion"].map(PUNTAJE_ETAPA).fillna(50)
cols_ej = ["edad", "ingreso", "etapa", "compromiso", "educacion", "cuenta_ahorro", "hipotecario", "credito_consumo"]
seg_ej = segmentar(d_ej, cols_ej)
print("Rand ajustado contra la segmentación base:", round(adjusted_rand_score(datos["segmento"], seg_ej), 3))
print()
print("Composición por ocupación con el eje etapa de vida:")
print(pd.crosstab(seg_ej, clientes["ocupacion"], normalize="index").round(2).to_string())
print()
print("Quiénes cambiaron de grupo (fila = segmento base, columna = nuevo):")
print(pd.crosstab(datos["segmento"], seg_ej))



Cambiar el segundo eje de la ocupación cambia la segmentación de forma moderada (Rand ajustado 0,71) y, otra vez, donde
corresponde: con el eje de etapa de vida los jubilados, que están solos en el extremo 100, pasan a formar un segmento
propio (98% jubilados), y los estudiantes, que en el eje de estabilidad compartían grupo con desempleados y dueñas de
casa, se reparten entre los segmentos jóvenes. El eje de etapa está además muy correlacionado con la edad, así que
aporta menos información nueva que el de estabilidad. La lección no es cuál mapeo es "el correcto",
sino que el mapeo elegido debe poder explicarse y que sus consecuencias se pueden medir.


In [ ]:

# Ejercicio 2: segmentar sin los productos de crédito y usarlos para describir
cols_sin_credito = ["edad", "ingreso", "estabilidad", "compromiso", "educacion", "cuenta_ahorro"]
seg_sin = segmentar(datos, cols_sin_credito)
clientes["segmento_sin_credito"] = seg_sin
print("Rand ajustado contra la base:", round(adjusted_rand_score(datos["segmento"], seg_sin), 3))
print()
resumen = datos.assign(segmento_sin_credito=seg_sin).groupby("segmento_sin_credito").agg(
    clientes=("edad", "size"), edad=("edad", "mean"), ingreso=("ingreso", "mean"), estabilidad=("estabilidad", "mean"), compromiso=("compromiso", "mean"),
    pct_consumo=("credito_consumo", "mean"), pct_hipotecario=("hipotecario", "mean"))
print(resumen.round(2).to_string())



Sin los productos de crédito en la segmentación, el segmento "con crédito de consumo" desaparece y los cinco grupos
quedan definidos por el perfil de la persona; el crédito de consumo se reparte entre ellos con poca variación, lo que
confirma que en esta base ese producto no sigue al perfil demográfico. Para el banco es la diferencia entre "¿quiénes
son mis clientes?" (segmentar sin productos) y "¿a quién le ofrezco qué?" (segmentar con ellos); las dos preguntas son
legítimas y conviene saber cuál se está respondiendo.


In [ ]:

# Ejercicio 3: estabilidad entre dos mitades
rng = np.random.default_rng(SEMILLA)
idx = rng.permutation(len(Z))
a, b = idx[: len(Z) // 2], idx[len(Z) // 2:]
ma = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z.iloc[a])
mb = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit(Z.iloc[b])
asignacion_desde_a = ma.predict(Z.iloc[b])          # los clientes de la mitad b, asignados a los centroides de a
print("Rand ajustado entre la segmentación propia de la mitad b y la heredada de la mitad a:",
      round(adjusted_rand_score(mb.labels_, asignacion_desde_a), 3))



Un índice de Rand ajustado alto entre las dos mitades significa que los segmentos no dependen de qué clientes tocaron
en la muestra: son un rasgo de la base, no del azar. Es la prueba de estabilidad que de verdad importa, porque una
segmentación que cambia con cada extracción no sirve para diseñar campañas que duran meses. Junto con la sensibilidad
a los supuestos de la sección 6.2, forma el par de preguntas que hay que responder antes de entregar una segmentación:
¿depende de la muestra? y ¿depende de mis decisiones, y de cuáles?
